# Імпорти та налаштування

In [ ]:
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import loguniform

from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import (
    RandomizedSearchCV,
    StratifiedKFold,
    cross_val_score,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
FIGURES_DIR = "figures"

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["figure.dpi"] = 120
sns.set_style("whitegrid")

print("Імпорти завершено успішно.")

# Завантаження даних та відтворення baseline з Лабораторної 1

Використовуємо той самий датасет, той самий split і той самий `StandardScaler`, щоб
числа baseline збіглися 1:1 з Лабою 1 та слугували чесною точкою відліку.

In [ ]:
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")
target_names = data.target_names

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train: {X_train.shape[0]}, Test: {X_test.shape[0]}, ознак: {X_train.shape[1]}")

In [ ]:
def evaluate(model, X_tr, y_tr, X_te, y_te):
    """Тренує модель і повертає словник ключових метрик на test."""
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    return {
        "accuracy": accuracy_score(y_te, y_pred),
        "precision": precision_score(y_te, y_pred, average="weighted"),
        "recall": recall_score(y_te, y_pred, average="weighted"),
        "f1": f1_score(y_te, y_pred, average="weighted"),
        "y_pred": y_pred,
    }


baseline_lr = evaluate(
    LogisticRegression(random_state=RANDOM_STATE, max_iter=10000),
    X_train_scaled, y_train, X_test_scaled, y_test,
)
baseline_rf = evaluate(
    RandomForestClassifier(random_state=RANDOM_STATE),
    X_train_scaled, y_train, X_test_scaled, y_test,
)

print("Baseline відтворено (має збігатися з Лабою 1):")
for name, m in [("LogisticRegression", baseline_lr), ("RandomForest", baseline_rf)]:
    print(f"  {name}: acc={m['accuracy']:.4f}  f1={m['f1']:.4f}")

# Методологія: Pipeline + StratifiedKFold + RandomizedSearchCV

- **Pipeline(StandardScaler → classifier)** гарантує, що скейлер fit'иться всередині
  кожного CV-фолду, а не один раз на весь train — це прибирає витік інформації.
- **StratifiedKFold(k=5, shuffle=True)** зберігає пропорцію класів (37/63) у кожному
  фолді; `shuffle=True` + `random_state=42` — відтворюваність.
- **RandomizedSearchCV** замість GridSearchCV: за той самий бюджет часу охоплює ширший
  простір. `n_iter=30`, `scoring='f1_weighted'` — вирівняно з метрикою порівняння.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
N_ITER = 30
SCORING = "f1_weighted"


def cv_stats(model_or_pipe, X_tr, y_tr):
    """Повертає (mean, std) F1-weighted крос-валідації."""
    scores = cross_val_score(model_or_pipe, X_tr, y_tr, cv=cv, scoring=SCORING, n_jobs=-1)
    return scores, scores.mean(), scores.std()


baseline_lr_cv_scores, baseline_lr_cv_mean, baseline_lr_cv_std = cv_stats(
    Pipeline([("scaler", StandardScaler()),
              ("clf", LogisticRegression(random_state=RANDOM_STATE, max_iter=10000))]),
    X_train, y_train,
)
baseline_rf_cv_scores, baseline_rf_cv_mean, baseline_rf_cv_std = cv_stats(
    Pipeline([("scaler", StandardScaler()),
              ("clf", RandomForestClassifier(random_state=RANDOM_STATE))]),
    X_train, y_train,
)

print(f"Baseline LR CV F1:  {baseline_lr_cv_mean:.4f} ± {baseline_lr_cv_std:.4f}")
print(f"Baseline RF CV F1:  {baseline_rf_cv_mean:.4f} ± {baseline_rf_cv_std:.4f}")

# Підбір гіперпараметрів: Logistic Regression

In [ ]:
lr_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(random_state=RANDOM_STATE, max_iter=10000)),
])

lr_param_dist = [
    # l2 + обидва solver
    {"clf__C": loguniform(1e-3, 1e2),
     "clf__penalty": ["l2"],
     "clf__solver": ["liblinear", "saga"]},
    # l1 + обидва solver
    {"clf__C": loguniform(1e-3, 1e2),
     "clf__penalty": ["l1"],
     "clf__solver": ["liblinear", "saga"]},
]

lr_search = RandomizedSearchCV(
    lr_pipe, lr_param_dist, n_iter=N_ITER, cv=cv,
    scoring=SCORING, n_jobs=-1, random_state=RANDOM_STATE, refit=True,
)
lr_search.fit(X_train, y_train)

print(f"Best params (LR): {lr_search.best_params_}")
print(f"Best CV F1 (LR):  {lr_search.best_score_:.4f}")

In [ ]:
y_pred_lr_opt = lr_search.predict(X_test)
optimized_lr = {
    "accuracy": accuracy_score(y_test, y_pred_lr_opt),
    "precision": precision_score(y_test, y_pred_lr_opt, average="weighted"),
    "recall": recall_score(y_test, y_pred_lr_opt, average="weighted"),
    "f1": f1_score(y_test, y_pred_lr_opt, average="weighted"),
    "y_pred": y_pred_lr_opt,
}
optimized_lr_cv_scores, optimized_lr_cv_mean, optimized_lr_cv_std = cv_stats(
    lr_search.best_estimator_, X_train, y_train,
)
print(f"Optimized LR test F1: {optimized_lr['f1']:.4f}")
print(f"Optimized LR CV F1:   {optimized_lr_cv_mean:.4f} ± {optimized_lr_cv_std:.4f}")

# Підбір гіперпараметрів: Random Forest

In [ ]:
rf_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", RandomForestClassifier(random_state=RANDOM_STATE)),
])

rf_param_dist = {
    "clf__n_estimators": [100, 200, 300, 500],
    "clf__max_depth": [None, 5, 10, 20],
    "clf__min_samples_split": [2, 5, 10],
    "clf__min_samples_leaf": [1, 2, 4],
    "clf__max_features": ["sqrt", "log2"],
}

rf_search = RandomizedSearchCV(
    rf_pipe, rf_param_dist, n_iter=N_ITER, cv=cv,
    scoring=SCORING, n_jobs=-1, random_state=RANDOM_STATE, refit=True,
)
rf_search.fit(X_train, y_train)

print(f"Best params (RF): {rf_search.best_params_}")
print(f"Best CV F1 (RF):  {rf_search.best_score_:.4f}")

In [ ]:
y_pred_rf_opt = rf_search.predict(X_test)
optimized_rf = {
    "accuracy": accuracy_score(y_test, y_pred_rf_opt),
    "precision": precision_score(y_test, y_pred_rf_opt, average="weighted"),
    "recall": recall_score(y_test, y_pred_rf_opt, average="weighted"),
    "f1": f1_score(y_test, y_pred_rf_opt, average="weighted"),
    "y_pred": y_pred_rf_opt,
}
optimized_rf_cv_scores, optimized_rf_cv_mean, optimized_rf_cv_std = cv_stats(
    rf_search.best_estimator_, X_train, y_train,
)
print(f"Optimized RF test F1: {optimized_rf['f1']:.4f}")
print(f"Optimized RF CV F1:   {optimized_rf_cv_mean:.4f} ± {optimized_rf_cv_std:.4f}")

# Classification report (optimized)

In [ ]:
print("=" * 60)
print("LOGISTIC REGRESSION (optimized)")
print("=" * 60)
print(classification_report(y_test, y_pred_lr_opt, target_names=target_names))

print("=" * 60)
print("RANDOM FOREST (optimized)")
print("=" * 60)
print(classification_report(y_test, y_pred_rf_opt, target_names=target_names))

# Зведена таблиця: Baseline vs Optimized

In [ ]:
rows = [
    ("Logistic Regression — baseline",  baseline_lr,  baseline_lr_cv_mean,  baseline_lr_cv_std),
    ("Logistic Regression — optimized", optimized_lr, optimized_lr_cv_mean, optimized_lr_cv_std),
    ("Random Forest — baseline",        baseline_rf,  baseline_rf_cv_mean,  baseline_rf_cv_std),
    ("Random Forest — optimized",       optimized_rf, optimized_rf_cv_mean, optimized_rf_cv_std),
]
summary = pd.DataFrame([{
    "Модель":    name,
    "Accuracy":  m["accuracy"],
    "Precision": m["precision"],
    "Recall":    m["recall"],
    "F1":        m["f1"],
    "CV F1 mean": cv_mean,
    "CV F1 std":  cv_std,
} for name, m, cv_mean, cv_std in rows]).set_index("Модель").round(4)
print(summary)

# Графіки

In [ ]:
# Boxplot CV-скорів: baseline vs optimized для обох моделей
fig, ax = plt.subplots(figsize=(8, 5))
cv_box_data = [
    baseline_lr_cv_scores, optimized_lr_cv_scores,
    baseline_rf_cv_scores, optimized_rf_cv_scores,
]
labels = ["LR\nbaseline", "LR\noptimized", "RF\nbaseline", "RF\noptimized"]
bp = ax.boxplot(cv_box_data, tick_labels=labels, patch_artist=True)
colors = ["#3498db", "#2ecc71", "#e67e22", "#e74c3c"]
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.set_ylabel("F1-weighted (CV fold)")
ax.set_title("Крос-валідаційні скори: Baseline vs Optimized (k=5)")
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/cv_scores_comparison.png", bbox_inches="tight")
plt.show()

In [ ]:
# Bar chart: порівняння метрик baseline vs optimized
metrics_names = ["Accuracy", "Precision", "Recall", "F1"]
x = np.arange(len(metrics_names))
width = 0.2

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - 1.5 * width, [baseline_lr[k.lower()]  for k in metrics_names], width, label="LR baseline",  color="#3498db")
ax.bar(x - 0.5 * width, [optimized_lr[k.lower()] for k in metrics_names], width, label="LR optimized", color="#2ecc71")
ax.bar(x + 0.5 * width, [baseline_rf[k.lower()]  for k in metrics_names], width, label="RF baseline",  color="#e67e22")
ax.bar(x + 1.5 * width, [optimized_rf[k.lower()] for k in metrics_names], width, label="RF optimized", color="#e74c3c")
ax.set_xticks(x)
ax.set_xticklabels(metrics_names)
ax.set_ylim(0.9, 1.0)
ax.set_ylabel("Значення")
ax.set_title("Метрики на test: Baseline vs Optimized")
ax.legend(loc="lower right", ncols=2)
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/metrics_baseline_vs_optimized.png", bbox_inches="tight")
plt.show()

In [ ]:
# Confusion matrices для оптимізованих моделей
for name, y_pred, filename in [
    ("Logistic Regression (optimized)", y_pred_lr_opt, "confusion_matrix_lr_opt.png"),
    ("Random Forest (optimized)",       y_pred_rf_opt, "confusion_matrix_rf_opt.png"),
]:
    fig, ax = plt.subplots(figsize=(5, 4))
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=target_names, yticklabels=target_names, ax=ax)
    ax.set_title(f"Confusion Matrix — {name}")
    ax.set_xlabel("Передбачений клас")
    ax.set_ylabel("Справжній клас")
    plt.tight_layout()
    plt.savefig(f"{FIGURES_DIR}/{filename}", bbox_inches="tight")
    plt.show()

# Підсумкові результати у JSON (для README)

In [ ]:
results = {
    "baseline": {
        "lr": {k: baseline_lr[k] for k in ["accuracy", "precision", "recall", "f1"]},
        "rf": {k: baseline_rf[k] for k in ["accuracy", "precision", "recall", "f1"]},
        "lr_cv": {"mean": baseline_lr_cv_mean, "std": baseline_lr_cv_std},
        "rf_cv": {"mean": baseline_rf_cv_mean, "std": baseline_rf_cv_std},
    },
    "optimized": {
        "lr": {k: optimized_lr[k] for k in ["accuracy", "precision", "recall", "f1"]},
        "rf": {k: optimized_rf[k] for k in ["accuracy", "precision", "recall", "f1"]},
        "lr_cv": {"mean": optimized_lr_cv_mean, "std": optimized_lr_cv_std},
        "rf_cv": {"mean": optimized_rf_cv_mean, "std": optimized_rf_cv_std},
        "lr_best_params": {k: (v if isinstance(v, (int, float, str, bool)) else str(v))
                           for k, v in lr_search.best_params_.items()},
        "rf_best_params": {k: (v if isinstance(v, (int, float, str, bool)) else str(v))
                           for k, v in rf_search.best_params_.items()},
    },
}
print(json.dumps(results, indent=2, ensure_ascii=False))